# Regressione delta_skill_tas vs delta_cover (disegno ibrido)

Domanda: il "miglioramento" della rappresentazione della cover effettiva
(SENS, forzata con le osservazioni, vs CTRL, modello di vegetazione dinamico)
e' associato al miglioramento dello skill della temperatura superficiale?

I due notebook precedenti hanno ciascuno un difetto per rispondere a questa
domanda:

- **01 (adattato)**: `delta_cover` e' valido (non circolare), ma `delta_tas` e'
  solo una differenza di anomalie SENS-CTRL — misura covarianza tra le due
  variabili, non un genuino miglioramento di skill. La relazione inoltre
  potrebbe essere non lineare, mentre la regressione usata e' lineare.
- **02 (letterale)**: `delta_tas` (skill vs ERA5) e' un genuino miglioramento
  di skill, ma `delta_cover` (skill vs "obs"=cover_SENS) e' circolare: SENS e'
  per costruzione ottimo, CTRL per costruzione peggiore, non puo' essere
  altrimenti — non e' una variabile che varia in modo informativo.

**Questo notebook prende il pezzo valido di ciascuno**:

- `X = delta_cover` (come 01: `cover_SENS - cover_CTRL`, non circolare).
- `Y = delta_skill_tas` (come 02, ma SOLO il lato temperatura, dove ERA5 e' un
  riferimento osservativo vero e indipendente — genuino miglioramento di skill).

Niente skill-vs-obs per la cover: eliminato il pezzo circolare del tutto.

Per la non linearita': accanto alla regressione lineare (Pearson: slope, r, p)
si calcola anche la correlazione di rango di **Spearman** (rho, p), che cattura
relazioni monotone anche non lineari. Prodotte due mappe (Pearson e Spearman,
ciascuna con la propria significativita') e uno scatter con entrambe le
statistiche in legenda.

Box Siberia (54-70N, 88-110E) invariato per ora — da adattare in base al
risultato della mappa 2D, come discusso.


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
era_var = '2t'
variables = ['cvh', 'cvl']
SAVE_PATH = str(FIG_DIR)


In [ ]:
# La logica di calcolo sta in cover_tas_lib.py (stesso modulo dei notebook 01/02,
# nuova funzione run_one_hybrid). Processi spawn freschi (ogni task in un
# processo nuovo), stesso pattern robusto adottato in questa sessione.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_hybrid, LEADS


In [ ]:
# --- DIAGNOSTICA: distribuzione di std(delta_cover), da eseguire PRIMA del
# calcolo completo per scegliere una soglia fissa informata (i valori di slope
# superavano 7000 anche col taglio al percentile 10 -> la maggior parte dei
# pixel ha varianza bassa, serve un numero scelto sui dati reali, non un
# percentile relativo).
from cover_tas_lib import debug_cover_variance
debug_cover_variance(exp_ctrl, exp_sens, 'cvh')
print()
debug_cover_variance(exp_ctrl, exp_sens, 'cvl')


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, era_var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_hybrid, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
